In [ ]:
pip install mp_api

In [ ]:
YOUR_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual Materials Project API key

In [ ]:
#Used to download data from the Materials Project database using the Materials Project API. The script below queries the database for materials containing Silicon (Si) and Sodium (Na), retrieves specific fields, and saves the results to a JSON file.



from mp_api.client import MPRester
import json

with MPRester(YOUR_API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        #elements=["Si", "Na"],
        formula = ["TiN"],
    #   fields=["material_id", 'elements', 'nelements', 'composition', "band_gap", "volume", "builder_meta", 'nsites', 'composition_reduced', 'formula_pretty', 'formula_anonymous', 'chemsys', 'density', 'density_atomic', 'symmetry', 'property_name', 'deprecated', 'deprecation_reasons', 'last_updated', 'origins', 'warnings', 'structure', 'task_ids', 'uncorrected_energy_per_atom', 'energy_per_atom', 'formation_energy_per_atom', 'energy_above_hull', 'is_stable', 'equilibrium_reaction_energy_per_atom', 'decomposes_to', 'xas', 'grain_boundaries', 'cbm', 'vbm', 'efermi', 'is_gap_direct', 'is_metal', 'es_source_calc_id', 'bandstructure', 'dos', 'dos_energy_up', 'dos_energy_down', 'is_magnetic', 'ordering', 'total_magnetization', 'total_magnetization_normalized_vol', 'total_magnetization_normalized_formula_units', 'num_magnetic_sites', 'num_unique_magnetic_sites', 'types_of_magnetic_species', 'bulk_modulus', 'shear_modulus', 'universal_anisotropy', 'homogeneous_poisson', 'e_total', 'e_ionic', 'e_electronic', 'n', 'e_ij_max', 'weighted_surface_energy_EV_PER_ANG2', 'weighted_surface_energy', 'weighted_work_function', 'surface_anisotropy', 'shape_factor', 'has_reconstructed', 'possible_species', 'has_props', 'theoretical', 'database_Ids']
    # ) #ALL field tags
        fields=["material_id", 'elements', 'nelements', 'composition', "band_gap", "volume", 'density', 'density_atomic', 'symmetry', 'uncorrected_energy_per_atom', 'energy_per_atom', 'formation_energy_per_atom', 'energy_above_hull', 'bulk_modulus', 'shear_modulus' ]
    )

if docs:
    output_filename = "materials_project_data_TiN.json" # Using .json extension for clarity
    with open(output_filename, 'w') as file:
        # docs is likely a list of dictionaries. Convert it to a JSON string for writing.
        file.write(json.dumps([doc.dict() for doc in docs], indent=4))
    print(f"Materials Project data successfully written to {output_filename}")
else:
    print("No data found for the specified query.")

In [ ]:
#Helps in retrieving band structure data for a specific material from the Materials Project database and saving it to a CSV file. The script uses the Materials Project API to fetch the band structure for a given material ID, extracts relevant information such as energies, k-points, and Fermi level, and organizes this data into a structured format before saving it.


import pandas as pd
from mp_api.client import MPRester
from pymatgen.electronic_structure.core import Spin

# =========================
# USER INPUT
# =========================
MP_ID = "mp-149"   # Example: Silicon

# =========================
# FETCH BAND STRUCTURE
# =========================
with MPRester(YOUR_API_KEY) as mpr:
    bs = mpr.get_bandstructure_by_material_id(MP_ID)

# =========================
# BASIC INFO
# =========================
kpoints = bs.kpoints
k_dist = bs.distance
k_frac = [kp.frac_coords for kp in kpoints]

bands = bs.bands
efermi = bs.efermi   # 🔑 critical for alignment

print(f"Fermi Energy (eV): {efermi}")

# =========================
# BUILD DATA
# =========================
data = []

def process_spin(energies, spin_label):
    nbands, nkpoints = energies.shape
    
    for i in range(nkpoints):
        for j in range(nbands):
            row = {
                "spin": spin_label,
                "k_index": i,
                "band_index": j,
                "kx": k_frac[i][0],
                "ky": k_frac[i][1],
                "kz": k_frac[i][2],
                "k_distance": k_dist[i],
                "energy_eV": energies[j][i],
                "energy_minus_fermi_eV": energies[j][i] - efermi   # ✅ aligned
            }
            data.append(row)

# =========================
# HANDLE SPIN CASES
# =========================
if Spin.up in bands:
    process_spin(bands[Spin.up], "up")

if Spin.down in bands:
    process_spin(bands[Spin.down], "down")

# =========================
# SAVE CSV
# =========================
df = pd.DataFrame(data)

output_file = f"{MP_ID}_bandstructure_fermi_aligned.csv"
df.to_csv(output_file, index=False)

print(f"Saved file: {output_file}")

In [ ]:
from mp_api.client import MPRester
import json
import pandas as pd

API_KEY = "YOUR_API_KEY"

# =========================
# USER INPUT (EDIT HERE)
# =========================
use_formula = True      # True → use formula search
use_elements = True     # True → use element search

formula_query = ["Si"]        # Example: ["TiN"]
elements_query = ["Si"]       # Example: ["Ti", "N"]

# =========================
# COMMON FIELDS
# =========================
fields = [
    "material_id", "elements", "nelements", "composition",
    "formula_pretty",
    "band_gap", "volume",
    "density", "density_atomic",
    "symmetry",
    "uncorrected_energy_per_atom",
    "energy_per_atom", "formation_energy_per_atom",
    "energy_above_hull",
    "bulk_modulus", "shear_modulus",
    "is_stable", "is_metal"
]

# =========================
# FUNCTION TO SAVE DATA
# =========================
def save_data(docs, prefix):
    if not docs:
        print(f"No data found for {prefix}")
        return

    data = [doc.dict() for doc in docs]

    # -------- JSON --------
    json_file = f"{prefix}.json"
    with open(json_file, "w") as f:
        json.dump(data, f, indent=4)
    print(f"Saved JSON: {json_file}")

    # -------- EXCEL --------
    df = pd.json_normalize(data)

    # Clean elements
    if "elements" in df.columns:
        df["elements"] = df["elements"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else x
        )

    # Clean composition
    if "composition" in df.columns:
        df["composition"] = df["composition"].astype(str)

    # Flatten symmetry
    if "symmetry.symbol" in df.columns:
        df.rename(columns={"symmetry.symbol": "spacegroup_symbol"}, inplace=True)

    if "symmetry.crystal_system" in df.columns:
        df.rename(columns={"symmetry.crystal_system": "crystal_system"}, inplace=True)

    excel_file = f"{prefix}.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"Saved Excel: {excel_file}")


# =========================
# MAIN QUERY EXECUTION
# =========================
with MPRester(YOUR_API_KEY) as mpr:

    # -------- FORMULA SEARCH --------
    if use_formula:
        docs_formula = mpr.materials.summary.search(
            formula=formula_query,
            fields=fields
        )
        save_data(docs_formula, "MP_formula_search")

    # -------- ELEMENT SEARCH --------
    if use_elements:
        docs_elements = mpr.materials.summary.search(
            elements=elements_query,
            fields=fields
        )
        save_data(docs_elements, "MP_element_search")